# Section 1: Why Passive RAG Breaks

*Duration: 15 minutes*

---

The Escalation Lab established a pattern: start with the simplest possible system, measure what it gets wrong, and only add complexity when the evidence justifies it.

That pattern produced a complete pipeline. A question arrives. The retriever pulls the top-k chunks from a vector store. The model reads those chunks and produces an answer. Every time. In that order.

If you worked through the Escalation Lab, you have seen this pipeline in action. If you are joining here directly, the pre-built outputs in `../prebuilt/` give you everything you need to run this section without having completed it first.

Either way, this section starts from the same place: a RAG pipeline that works reasonably well, and two questions it consistently gets wrong.

This section is not about improving retrieval or the model. It is about understanding what the architecture itself cannot do, and why that gap matters before we build anything new.

## 1.1 Re-Introducing the Baseline

The evaluation artifact below was produced at the end of the Escalation Lab. It captures the 10 evaluation questions and their final pass/fail classifications after RAG, Best-of-N, and fine-tuning were all applied.

If you completed the Escalation Lab and want to use your own results, point the path below at your generated file. Otherwise, the pre-built version is ready to go.

In [1]:
import json

EVAL_PATH = "../prebuilt/eval_results.json"  # swap path if using your own output

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

results = eval_data["results"]

passes   = [r for r in results if r["classification"] == "pass"]
failures = [r for r in results if r["classification"] != "pass"]

print(f"Total questions : {len(results)}")
print(f"Passes          : {len(passes)}")
print(f"Failures        : {len(failures)}")

Total questions : 10
Passes          : 8
Failures        : 2


You should see 8 passes and 2 remaining failures.

Those two questions survived every improvement the Escalation Lab applied: better chunking, RAG, Best-of-N sampling, and LoRA fine-tuning. They are not random noise. They are the pipeline telling you something specific about its own architecture.

If you are joining this lab without completing the Escalation Lab, here is what you need to know about these results:

- The corpus is the **Basic Fantasy RPG rulebook**, ingested and chunked using Docling
- The 10 questions cover character class abilities, combat rules, and equipment
- The pipeline uses cosine similarity retrieval against a ChromaDB vector store
- The model is `granite-3-2-8b-instruct` served via the Red Hat MaaS endpoint
- 8 of the 10 questions are answered correctly after all optimizations. 2 are not.

## 1.2 Looking at the Failures

Before categorizing anything, read the failures directly. Look at each question and its expected answer.

Ask yourself: why might a retrieve-then-answer pipeline fail to produce the correct response?

In [2]:
print("Remaining failures")
print("=" * 60)

for r in failures:
    print(f"\nQuestion       : {r['question']}")
    print(f"Expected       : {r['expected']}")
    print(f"Category       : {r['category']}")
    print(f"Classification : {r['classification']}")
    print("-" * 60)

Remaining failures

Question       : Why can't Elves roll higher than a d6 for hit points?
Expected       : Elves use a d6 for hit points because that is the hit die assigned to the Elf combination class in Basic Fantasy RPG.
Category       : terminology
Classification : fail
------------------------------------------------------------

Question       : Can a character wear leather armor and cast spells?
Expected       : Magic-Users and Elves cannot cast spells while wearing armor. Clerics can wear armor and cast spells.
Category       : implicit_reasoning
Classification : fail
------------------------------------------------------------


Read both outputs carefully before moving on.

In most cases, one of two things is happening:

- The chunks do not contain the information needed to answer the question
- The chunks contain the information, but answering correctly requires combining facts the model retrieved and then did not use

These are different problems. The pipeline cannot tell them apart. That is the architectural gap this lab addresses.

## 1.3 Naming the Three Failure Types

The failures you just read map to one of three categories. Naming them precisely matters because each one points to a different fix.

---

### Irrelevant Retrieval

The retriever returned chunks that are topically adjacent but do not contain the answer. The model received context that looked relevant and answered from it anyway. The answer is wrong because the input was wrong, and the pipeline had no way to detect that before committing to a response.

The retriever has no relevance threshold. It returns the closest chunks it has, relevant or not, and the model answers regardless.

---

### Implicit Reasoning

The answer exists in the corpus. The retriever found the right chunks. But the question requires combining two facts from different sections of the document, and the model did not make that connection.

This is not a retrieval failure. It is a reasoning gap that retrieval alone cannot close. Adding more training data will not fix it. Changing the chunking strategy will not fix it. The pipeline needs a different control structure.

---

### Out-of-Scope Questions

The answer does not exist anywhere in the corpus. The correct response is: *"I do not have that information."*

A passive pipeline does not produce that response. It retrieves whatever is closest and answers from it, producing a confident, grounded-sounding, incorrect answer.

---

Fill in the classification below based on your inspection of the outputs above. You will come back to this table in Section 2.

In [3]:
# Complete this classification based on your inspection above.
# Failure types: "irrelevant_retrieval", "implicit_reasoning", "out_of_scope"

failure_classifications = [
    {
        "id": failures[0]["id"],
        "question": failures[0]["question"],
        "failure_type": "irrelevant_retrieval",  # <-- update this
        "evidence": ""                            # <-- one sentence explaining why
    },
    {
        "id": failures[1]["id"],
        "question": failures[1]["question"],
        "failure_type": "implicit_reasoning",     # <-- update this
        "evidence": ""                            # <-- one sentence explaining why
    }
]

print("Failure Classification")
print("=" * 60)
for fc in failure_classifications:
    print(f"\n  ID           : {fc['id']}")
    print(f"  Question     : {fc['question']}")
    print(f"  Failure type : {fc['failure_type']}")
    print(f"  Evidence     : {fc['evidence'] or '(not yet filled in)'}")

Failure Classification

  ID           : q02
  Question     : Why can't Elves roll higher than a d6 for hit points?
  Failure type : irrelevant_retrieval
  Evidence     : (not yet filled in)

  ID           : q03
  Question     : Can a character wear leather armor and cast spells?
  Failure type : implicit_reasoning
  Evidence     : (not yet filled in)


Irrelevant retrieval has two distinct root causes that look identical from the pipeline's perspective. The query may be poorly formed — using terminology that does not match the source documents, or asking a question at the wrong level of specificity. Alternatively, the corpus itself may be incomplete or poorly chunked — the answer simply is not there, or it was split across chunk boundaries in a way that destroys its meaning. Both produce weak retrieval scores. Both will fool the passive pipeline into answering from whatever it found.

The distinction matters because the two causes require different interventions. The agent loop introduced in Section 2 addresses the query side: it evaluates retrieval quality, rewrites the query, and tries again. That works when the answer exists in the corpus but the original phrasing missed it. It cannot address the corpus side. If the document does not contain the answer, or if the chunking process destroyed the structure needed to answer the question, no amount of query rewriting will find it.

Setting this expectation now is important. The agent loop is not a universal fix for retrieval failures. It is a fix for the subset of failures caused by a mismatch between the question and the retrieval index. Corpus quality is a separate problem that sits upstream, and it requires separate tools.

> **Facilitator note:** Participants often want to say "the model should just know better." Redirect that instinct. The failure type determines the fix. Irrelevant retrieval is a pipeline problem. Implicit reasoning requires a different control structure. Out-of-scope questions require the system to recognize when it cannot answer. None of those fixes are "train the model harder."

## 1.4 The Architectural Problem

A passive RAG pipeline is built around one assumption: retrieval will return something useful.

When that assumption holds, the pipeline works well. When it does not, the pipeline has no recovery path. It retrieved. It answered. It moved on.

This is not a flaw in the retriever or the model. It is a structural property of the architecture. The pipeline does not inspect the retrieval result before deciding whether to answer. It does not ask whether the retrieved context is relevant. It does not consider whether the question falls outside the scope of the corpus. It executes the same sequence every time.

Connect to the MaaS endpoint and run the passive pipeline on one of the failing questions so you can see the behavior directly.

### 1.4.1 Install OpenAI

The cell below installs the OpenAI Python client library. The MaaS endpoint we use exposes an OpenAI-compatible API, so this library lets us call the hosted Granite model with the same interface you would use for any OpenAI-compatible service.

In [5]:
! pip install openai -q

### 1.4.2 Load configuration

The cell below adds the parent directory to the Python path and imports the shared `config` module. This module provides the API key, endpoint URL, and model ID so that every notebook in the lab connects to the same MaaS-hosted Granite model without duplicating credentials.

In [6]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import config
from openai import OpenAI

client = OpenAI(api_key=config.API_KEY, base_url=config.ENDPOINT_BASE)

print(f"Connected to : {config.ENDPOINT_BASE}")
print(f"Model        : {config.MODEL_ID}")

Connected to : https://litellm-prod.apps.maas.redhatworkshops.io/v1
Model        : granite-3-2-8b-instruct


### 1.4.3 Define passive_rag function

The cell below defines `passive_rag`, a minimal version of the retrieve-then-answer pipeline from the Escalation Lab. It sends the question to the model with an empty context block — simulating what happens when the retriever returns nothing useful. This lets you observe the core architectural problem directly: the model will still produce an answer, even when it has no supporting evidence to draw from.

In [ ]:
def passive_rag(question, model_client, model_id):
    """
    The pipeline as built in the Escalation Lab — without retrieval context.

    In production, a retriever would supply chunks here. Without a vector
    store available, this demonstrates the model's raw behavior: it answers
    regardless of whether it has the information.
    """
    prompt = f"""You are a rules assistant for Basic Fantasy RPG.
Use only the context below to answer the question.
If the context does not contain enough information, say so.

Context:
(no context retrieved)

Question: {question}"""

    response = model_client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content


question = failures[0]["question"]
answer = passive_rag(question, client, config.MODEL_ID)

print(f"Question      : {question}")
print(f"Expected      : {failures[0]['expected']}")
print(f"\nModel answered: {answer}")

The pipeline ran. It retrieved. It answered.

It had no mechanism to detect that the retrieval was insufficient before producing that answer. The model was not malfunctioning. It did exactly what the architecture asked of it. The architecture asked the wrong thing.

That is the problem the agent loop solves. The loop gives the model a decision to make before it commits to an answer. The retriever does not go away. It becomes one option among several, and the model decides when to use it.

---

> **FIELD TAKEAWAY**
>
> A passive pipeline that always retrieves and always answers will always hallucinate on questions where retrieval fails. The fix is not a better model. It is a control structure that can inspect, decide, and choose a different path.

---

## What Comes Next

Section 2 introduces the agent loop: the control structure that gives the model a decision to make before it commits to an answer.

The retriever does not go away. It becomes one option among several, and the model decides when to use it.

Move to `02_The_Agent_Loop.ipynb`.